# Instance audit: projects → Snowflake/OpenAI/code envs

This notebook scans the current DSS instance and writes a dataset named `INSTANCE_PROJECT_USAGE` in this project.

- Read-only against scanned projects
- Requires permissions to list/access the projects


In [ ]:
import pandas as pd
import dataiku

from dku_project_bulk_move.instance_audit import collect_instance_project_usage

OUTPUT_DATASET = "INSTANCE_PROJECT_USAGE"

client = dataiku.api_client()
project_key = dataiku.default_project_key()
project = client.get_project(project_key)

# Ensure the output dataset exists and is managed (so overwriting is allowed)
try:
    ds_handle = project.get_dataset(OUTPUT_DATASET)
    ds_raw = ds_handle.get_settings().get_raw()
except Exception as e:
    raise Exception(
        f"Dataset '{OUTPUT_DATASET}' not found (or not accessible). Pull the project from Git or create it first."
    ) from e

if str(ds_raw.get("type")) != "Filesystem" or not bool(ds_raw.get("managed")):
    raise Exception(
        f"Dataset '{OUTPUT_DATASET}' must be a managed Filesystem dataset. Current: type={ds_raw.get('type')}, managed={ds_raw.get('managed')}"
    )

# Collect + write
rows = collect_instance_project_usage(client)
df = pd.DataFrame.from_records([r.as_dict() for r in rows])
dataiku.Dataset(OUTPUT_DATASET).write_with_schema(df)

{"rows": len(df), "dataset": OUTPUT_DATASET, "projectKey": project_key}


In [ ]:
# Quick preview
dataiku.Dataset(OUTPUT_DATASET).get_dataframe().head(20)